# Drift mùa vụ và drift COVID trên NYC TLC

Bốn tháng `2019-07`, `2019-12`, `2020-02`, `2020-04`, mỗi tháng mẫu 100 000 chuyến (`seed=0`). Điền 6 hàm trong các cell chấm điểm; cell `explore` chỉ để xem.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import train_test_split

MONTHS = {
    "2019-07": "nyc-tlc",
    "2019-12": "nyc-tlc",
    "2020-02": "nyc-tlc-2020",
    "2020-04": "nyc-tlc-2020",
}
SAMPLE_N = 100_000
SEED = 0
COLUMNS = ["tpep_pickup_datetime", "tpep_dropoff_datetime", "trip_distance", "fare_amount", "passenger_count", "PULocationID"]
FEATURES = ["trip_distance", "fare_amount", "hour"]
MODEL_FEATURES = ["trip_distance", "hour", "PULocationID"]

In [ ]:
def load_sample(path, n=SAMPLE_N, seed=SEED):
    return pd.read_parquet(path, columns=COLUMNS).sample(n=n, random_state=seed)

In [ ]:
def prepare(df):
    df = df.copy()
    df["duration_min"] = (df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]).dt.total_seconds() / 60
    df["hour"] = df["tpep_pickup_datetime"].dt.hour
    keep = (df["duration_min"] > 0) & (df["duration_min"] < 180) & (df["trip_distance"] > 0) & (df["fare_amount"] > 0)
    return df[keep].reset_index(drop=True)

In [ ]:
def psi(reference, current, bins=10, eps=1e-4):
    reference = np.asarray(reference, dtype=float)
    current = np.asarray(current, dtype=float)
    reference = reference[~np.isnan(reference)]
    current = current[~np.isnan(current)]
    edges = np.unique(np.quantile(reference, np.linspace(0, 1, bins + 1)))
    edges[0], edges[-1] = -np.inf, np.inf
    ref_share = np.clip(np.histogram(reference, edges)[0] / len(reference), eps, None)
    cur_share = np.clip(np.histogram(current, edges)[0] / len(current), eps, None)
    return float(np.sum((cur_share - ref_share) * np.log(cur_share / ref_share)))

In [ ]:
def psi_by_column(reference, current, columns=FEATURES):
    return {c: psi(reference[c], current[c]) for c in columns}

In [ ]:
def missing_share(df, column):
    return float(df[column].isna().mean())

In [ ]:
def eta_mae(reference, current, seed=SEED):
    train, holdout = train_test_split(reference, test_size=0.2, random_state=seed)
    model = HistGradientBoostingRegressor(random_state=seed, max_iter=100)
    model.fit(train[MODEL_FEATURES], train["duration_min"])
    mae_holdout = float(np.abs(model.predict(holdout[MODEL_FEATURES]) - holdout["duration_min"]).mean())
    mae_current = float(np.abs(model.predict(current[MODEL_FEATURES]) - current["duration_min"]).mean())
    return mae_holdout, mae_current

In [ ]:
# explore: tải dữ liệu nếu chưa có và đọc bốn tháng
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure

data = {}
for month, dataset in MONTHS.items():
    path = ensure(dataset) / f"yellow_tripdata_{month}.parquet"
    data[month] = prepare(load_sample(path))
    print(month, len(data[month]), "dòng sau lọc")

In [ ]:
# explore: PSI từng feature cho hai cặp tháng
pairs = {"mùa vụ 2019-07 -> 2019-12": ("2019-07", "2019-12"), "COVID 2020-02 -> 2020-04": ("2020-02", "2020-04")}
table = pd.DataFrame({name: psi_by_column(data[a], data[b]) for name, (a, b) in pairs.items()})
print(table.round(4))
for name, (a, b) in pairs.items():
    print(name, "passenger_count NaN:", round(missing_share(data[a], "passenger_count"), 4), "->", round(missing_share(data[b], "passenger_count"), 4))

In [ ]:
# explore: chất lượng model ETA huấn luyện trên tháng đầu của mỗi cặp
for name, (a, b) in pairs.items():
    holdout, current = eta_mae(data[a], data[b])
    print(name, "MAE holdout", round(holdout, 3), "MAE tháng sau", round(current, 3), "tăng", f"{current / holdout - 1:.1%}")

In [ ]:
# explore: biểu đồ PSI hai cặp tháng
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

ax = table.plot.bar(rot=0)
ax.axhline(0.1, color="gray", linestyle="--")
ax.set_ylabel("PSI")
plt.tight_layout()
plt.savefig("psi_seasonal_vs_covid.png")

Làm phần trắc nghiệm trong app sau khi chạy notebook.